# PYNQ-Z2 ANC Project - Full ANC System Operation

This notebook will:
1. Test the complete ANC system with real audio input
2. Capture input (noisy) and output (cleaned) audio
3. Verify noise cancellation performance
4. Monitor step-size adaptation in real-time

## Prerequisites
- Complete `01_setup.ipynb` and `02_audio_codec_test.ipynb`
- Connect audio sources:
  - Primary audio (signal + noise) → Codec Left Channel
  - Reference noise (correlated) → Codec Right Channel
- Connect headphones/speakers to codec output
- Set SW0 to position 0 (ANC active mode)

## 1. Import Libraries and Load Overlay

In [ ]:
from pynq import Overlay
from pynq import MMIO
import numpy as np
import matplotlib.pyplot as plt
import time

# Load the bitstream
overlay = Overlay('anc_project.bit')
print("Bitstream loaded successfully")

## 2. Access ANC Core Registers

The ANC core should expose:
- Primary input (d[n])
- Reference noise input (x[n])
- Cleaned output (e[n])
- Active step-size shift (mu_shift)
- Data ready flag

In [ ]:
# Find ANC core IP
anc_core = None

for ip_name, ip_info in overlay.ip_dict.items():
    if 'anc' in ip_name.lower() or 'core' in ip_name.lower():
        anc_core = ip_name
        print(f"Found ANC core: {ip_name}")
        print(f"  Address: 0x{ip_info['phys_addr']:08X}")
        print(f"  Size: {ip_info['addr_range']} bytes")
        break

if anc_core:
    mmio = MMIO(overlay.ip_dict[anc_core]['phys_addr'], 
               overlay.ip_dict[anc_core]['addr_range'])
    print(f"\nMMIO object created for {anc_core}")
else:
    print("No ANC core found. Check the Vivado design.")

## 3. Configure System for ANC Mode

Ensure bypass switch is disabled (ANC active).

In [ ]:
# Find GPIO controller for bypass switch
bypass_gpio = None

for ip_name, ip_info in overlay.ip_dict.items():
    if 'gpio' in ip_name.lower():
        bypass_gpio = ip_name
        print(f"Found GPIO controller: {ip_name}")
        break

if bypass_gpio:
    # Set bypass to 0 (ANC active)
    # Note: Adjust GPIO configuration based on your design
    print("Configuring ANC mode (bypass = 0)...")
    print("Ensure SW0 is in position 0 on the board.")
else:
    print("Manual configuration required: Set SW0 to position 0.")

## 4. Monitor Initial State

Check the initial step-size and error values before audio input.

In [ ]:
if anc_core:
    # Define register offsets (adjust based on your design)
    MU_SHIFT_OFFSET = 0x00
    PRIMARY_IN_OFFSET = 0x04
    REF_NOISE_OFFSET = 0x08
    CLEANED_OUT_OFFSET = 0x0C
    DATA_READY_OFFSET = 0x10
    
    print("Reading initial ANC state...")
    print("="*50)
    
    try:
        mu_shift = mmio.read(MU_SHIFT_OFFSET)
        primary_in = mmio.read(PRIMARY_IN_OFFSET)
        ref_noise = mmio.read(REF_NOISE_OFFSET)
        cleaned_out = mmio.read(CLEANED_OUT_OFFSET)
        data_ready = mmio.read(DATA_READY_OFFSET)
        
        print(f"Step-size shift (mu): {mu_shift}")
        print(f"Primary input (d[n]): {primary_in}")
        print(f"Reference noise (x[n]): {ref_noise}")
        print(f"Cleaned output (e[n]): {cleaned_out}")
        print(f"Data ready: {data_ready}")
    except Exception as e:
        print(f"Error reading registers: {e}")
        print("Register offsets may need adjustment.")
else:
    print("Skipping initial state check - no ANC core found.")

## 5. Capture ANC Operation with Audio

Capture a sequence of samples showing the ANC in action.
This includes primary input, reference noise, and cleaned output.

In [ ]:
if anc_core:
    print("Capturing ANC operation (5 seconds)...")
    print("(Play audio with noise during this period)")
    print("="*50)
    
    NUM_SAMPLES = 240000  # ~5 seconds at 48.8 kHz
    primary_buffer = []
    ref_noise_buffer = []
    cleaned_buffer = []
    mu_shift_buffer = []
    
    start_time = time.time()
    sample_count = 0
    
    while sample_count < NUM_SAMPLES and time.time() - start_time < 6.0:
        try:
            data_ready = mmio.read(DATA_READY_OFFSET)
            
            if data_ready & 1:
                primary = mmio.read(PRIMARY_IN_OFFSET)
                ref_noise = mmio.read(REF_NOISE_OFFSET)
                cleaned = mmio.read(CLEANED_OUT_OFFSET)
                mu_shift = mmio.read(MU_SHIFT_OFFSET)
                
                # Convert to signed 16-bit
                if primary >= 32768:
                    primary = primary - 65536
                if ref_noise >= 32768:
                    ref_noise = ref_noise - 65536
                if cleaned >= 32768:
                    cleaned = cleaned - 65536
                
                primary_buffer.append(primary)
                ref_noise_buffer.append(ref_noise)
                cleaned_buffer.append(cleaned)
                mu_shift_buffer.append(mu_shift)
                sample_count += 1
                
                if sample_count % 50000 == 0:
                    print(f"Captured {sample_count} samples...")
        except Exception as e:
            pass
        time.sleep(0.0001)
    
    primary_buffer = np.array(primary_buffer)
    ref_noise_buffer = np.array(ref_noise_buffer)
    cleaned_buffer = np.array(cleaned_buffer)
    mu_shift_buffer = np.array(mu_shift_buffer)
    
    print(f"\nCapture complete. Total samples: {len(primary_buffer)}")
else:
    print("Skipping capture - no ANC core found.")

## 6. Visualize ANC Results

Plot the input (noisy) vs output (cleaned) audio.

In [ ]:
if anc_core and len(primary_buffer) > 0:
    # Plot first 5000 samples
    num_plot = min(5000, len(primary_buffer))
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
    
    # Primary input (noisy)
    ax1.plot(primary_buffer[:num_plot], label='Primary (Noisy)', alpha=0.7)
    ax1.set_title('Primary Input (Signal + Noise)')
    ax1.set_ylabel('Amplitude (16-bit)')
    ax1.legend()
    ax1.grid(True)
    
    # Cleaned output
    ax2.plot(cleaned_buffer[:num_plot], label='Cleaned Output', color='green', alpha=0.7)
    ax2.set_title('ANC Output (Cleaned Signal)')
    ax2.set_xlabel('Sample')
    ax2.set_ylabel('Amplitude (16-bit)')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

## 7. Calculate Noise Reduction Metrics

In [ ]:
if anc_core and len(primary_buffer) > 0:
    # Calculate RMS (Root Mean Square) for input and output
    rms_input = np.sqrt(np.mean(primary_buffer**2))
    rms_output = np.sqrt(np.mean(cleaned_buffer**2))
    
    # Calculate noise reduction in dB
    noise_reduction_db = 20 * np.log10(rms_input / rms_output) if rms_output > 0 else 0
    
    print("Noise Reduction Metrics:")
    print("="*50)
    print(f"RMS Input (Noisy):  {rms_input:.2f}")
    print(f"RMS Output (Clean): {rms_output:.2f}")
    print(f"Noise Reduction:    {noise_reduction_db:.2f} dB")
    
    if noise_reduction_db > 3:
        print("\n✓ Significant noise reduction detected")
    elif noise_reduction_db > 0:
        print("\n~ Moderate noise reduction")
    else:
        print("\n✗ No noise reduction (check signal correlation)")
else:
    print("No data for metrics calculation.")

## 8. Visualize Step-Size Adaptation

Plot how the step-size (mu_shift) changes over time during convergence.

In [ ]:
if anc_core and len(mu_shift_buffer) > 0:
    # Downsample for plotting
    downsample = 100
    mu_shift_plot = mu_shift_buffer[::downsample]
    
    plt.figure(figsize=(14, 4))
    plt.plot(mu_shift_plot)
    plt.title('Step-Size Adaptation (mu_shift) Over Time')
    plt.xlabel('Time (downsampled)')
    plt.ylabel('Step-Size Shift Value')
    plt.grid(True)
    
    # Add threshold lines
    plt.axhline(y=1, color='r', linestyle='--', label='Shift 1 (Fast)')
    plt.axhline(y=3, color='orange', linestyle='--', label='Shift 3 (Medium)')
    plt.axhline(y=6, color='g', linestyle='--', label='Shift 6 (Fine)')
    plt.legend()
    
    plt.show()
    
    print(f"Initial mu_shift: {mu_shift_buffer[0]}")
    print(f"Final mu_shift: {mu_shift_buffer[-1]}")
    print(f"Average mu_shift: {np.mean(mu_shift_buffer):.2f}")
else:
    print("No step-size data to display.")

## 9. Spectral Analysis

Compare the frequency spectrum of input vs output to see noise reduction.

In [ ]:
if anc_core and len(primary_buffer) > 0:
    from scipy import signal
    
    # Compute FFT
    n_fft = min(4096, len(primary_buffer))
    
    freqs = np.fft.rfftfreq(n_fft, d=1/48828)
    fft_input = np.abs(np.fft.rfft(primary_buffer[:n_fft]))
    fft_output = np.abs(np.fft.rfft(cleaned_buffer[:n_fft]))
    
    # Convert to dB
    fft_input_db = 20 * np.log10(fft_input + 1e-10)
    fft_output_db = 20 * np.log10(fft_output + 1e-10)
    
    plt.figure(figsize=(14, 6))
    plt.semilogx(freqs, fft_input_db, label='Input (Noisy)', alpha=0.7)
    plt.semilogx(freqs, fft_output_db, label='Output (Cleaned)', alpha=0.7)
    plt.title('Frequency Spectrum: Input vs Output')
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Magnitude (dB)')
    plt.legend()
    plt.grid(True)
    plt.xlim([100, 20000])
    plt.show()
else:
    print("No data for spectral analysis.")

## 10. Real-Time Monitoring

Monitor ANC performance in real-time for 10 seconds.

In [ ]:
if anc_core:
    print("Real-time ANC monitoring (10 seconds)...")
    print("="*50)
    
    for i in range(10):
        try:
            mu_shift = mmio.read(MU_SHIFT_OFFSET)
            primary = mmio.read(PRIMARY_IN_OFFSET)
            cleaned = mmio.read(CLEANED_OUT_OFFSET)
            
            # Convert to signed
            if primary >= 32768:
                primary = primary - 65536
            if cleaned >= 32768:
                cleaned = cleaned - 65536
            
            rms_in = abs(primary)
            rms_out = abs(cleaned)
            
            print(f"[{i}s] mu_shift: {mu_shift} | Input RMS: {rms_in:5d} | Output RMS: {rms_out:5d}")
        except:
            print(f"[{i}s] Unable to read registers")
        
        time.sleep(1)
    
    print("\nMonitoring complete.")
else:
    print("Skipping real-time monitoring - no ANC core found.")

## 11. Save Captured Audio for Analysis

Save the captured audio to WAV files for external analysis.

In [ ]:
if anc_core and len(primary_buffer) > 0:
    import wave
    
    def save_wav(filename, data, sample_rate=48000):
        with wave.open(filename, 'w') as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)  # 16-bit
            wav_file.setframerate(sample_rate)
            wav_file.writeframes(data.astype(np.int16).tobytes())
    
    save_wav('anc_input.wav', primary_buffer)
    save_wav('anc_output.wav', cleaned_buffer)
    
    print("Audio files saved:")
    print("  - anc_input.wav (noisy input)")
    print("  - anc_output.wav (cleaned output)")
else:
    print("No audio data to save.")

## 12. Summary

### Verification Checklist:
- [ ] ANC core accessible
- [ ] Bypass disabled (ANC mode active)
- [ ] Audio capture working
- [ ] Noise reduction observed (>3 dB)
- [ ] Step-size adaptation visible
- [ ] Spectral analysis shows noise reduction

### Performance Summary:
- Noise Reduction: (See cell 7 output)
- Final Step-Size: (See cell 8 output)

### Next Steps:
- Proceed to `04_performance_analysis.ipynb` for detailed convergence analysis
- Proceed to `05_bypass_mode.ipynb` to verify bypass functionality
- Adjust audio source parameters if noise reduction is insufficient